In [ ]:
# ============================================================================
# (1) Convert a BMG CLARIOstar plate export -> Quant_DNA_Input long format
# ============================================================================
# The reader writes a 96-well grid (rows A-H x cols 1-12) under a few header
# lines.  The rest of this notebook expects the tidy Qubit-style layout
# (Well, Content, Raw_Data, Concentration, Sample_Name), so convert first.
#
# Well -> concentration / name assignments live in the *_metadata.csv files,
# following the same convention as the other metadata files in absorbances/.

import os
import re
import numpy as np
import pandas as pd

REPO = os.getcwd()

PLATE_FILE     = os.path.join(REPO, "absorbances", "20261016_Ik_DNA_BR_soil_practice.CSV")
STANDARDS_META = os.path.join(REPO, "absorbances", "20261016_DNA_standards_metadata.csv")
SAMPLES_META   = os.path.join(REPO, "absorbances", "20261016_DNA_samples_metadata.csv")

output_dir = os.path.join(REPO, "concentrations")
os.makedirs(output_dir, exist_ok=True)

LONG_CSV = os.path.splitext(PLATE_FILE)[0] + "_quantinput.csv"


def read_bmg_plate(path):
    """Read a BMG CLARIOstar grid CSV into (tidy DataFrame, header metadata dict).

    The grid is located by its column-label line (",1,2,3,...,12,"); everything
    above it is treated as header, everything below as one plate row per line.
    """
    with open(path, "r", encoding="utf-8-sig", newline="") as fh:
        lines = [ln.rstrip("\r\n") for ln in fh]

    header_idx = next(i for i, ln in enumerate(lines)
                      if re.match(r"^\s*,\s*1\s*,\s*2\s*,", ln))

    columns = [int(c) for c in
               (c.strip() for c in lines[header_idx].split(",")[1:]) if c]

    records = []
    for ln in lines[header_idx + 1:]:
        cells = [c.strip() for c in ln.split(",")]
        if not cells or not re.fullmatch(r"[A-H]", cells[0]):
            continue
        for col, val in zip(columns, cells[1:]):
            records.append({"Well": f"{cells[0]}{col:02d}",
                            "Row": cells[0],
                            "Col": col,
                            "Raw_Data": pd.to_numeric(val, errors="coerce")})

    meta, notes = {}, []
    for ln in lines[:header_idx]:
        if not ln.strip(","):
            continue
        for field in ln.split(","):
            field = field.strip()
            if not field:
                continue
            if field.startswith("Path"):
                meta["Path"] = field.split(":", 1)[1].strip()
            elif ":" in field:
                k, v = field.split(":", 1)
                meta[k.strip()] = v.strip()
            else:
                notes.append(field)
    meta["Notes"] = " | ".join(notes)

    return pd.DataFrame(records), meta


def build_quant_input(plate, standards_meta, samples_meta):
    """Merge the plate grid with the well metadata into Quant_DNA_Input layout."""
    std = standards_meta.assign(Well=lambda d: d["Well"].str.strip()) \
                        .rename(columns={"Concentration": "Known_Concentration"})
    smp = samples_meta.assign(Well=lambda d: d["Well"].str.strip())

    df = plate.merge(std[["Well", "Known_Concentration"]], on="Well", how="left")
    df = df.merge(smp[["Well", "Sample_Name"]], on="Well", how="left")

    is_std = df["Known_Concentration"].notna()

    # Unlisted wells that still hold a reading are treated as unnamed samples.
    df["Content"] = ""
    df.loc[is_std,  "Content"] = ["Standard S%d" % i for i in range(1, is_std.sum() + 1)]
    df.loc[~is_std, "Content"] = ["Sample X%d"   % i for i in range(1, (~is_std).sum() + 1)]

    df["Sample_Name"] = df["Sample_Name"].fillna("").astype(str).str.strip()
    df.loc[is_std, "Sample_Name"] = ""
    unnamed = (~is_std) & (df["Sample_Name"] == "")
    df.loc[unnamed, "Sample_Name"] = df.loc[unnamed, "Well"]   # fall back to well ID

    df["Concentration"] = df["Known_Concentration"]            # blank for samples

    return df[["Well", "Row", "Col", "Content", "Raw_Data",
               "Concentration", "Sample_Name", "Known_Concentration"]]


plate, plate_meta = read_bmg_plate(PLATE_FILE)
standards_meta = pd.read_csv(STANDARDS_META)
samples_meta   = pd.read_csv(SAMPLES_META)

long_df = build_quant_input(plate, standards_meta, samples_meta)

# Quant_DNA_Input.csv-compatible file (exactly the 5 columns the original code reads)
long_df[["Well", "Content", "Raw_Data", "Concentration", "Sample_Name"]].to_csv(
    LONG_CSV, index=False)

print(f"Run     : {plate_meta.get('Test name')}  ({plate_meta.get('Date')} {plate_meta.get('Time')})")
print(f"ID1     : {plate_meta.get('ID1')}")
print(f"Signal  : {plate_meta.get('Notes')}")
print(f"Wells   : {len(long_df)}  "
      f"({long_df['Known_Concentration'].notna().sum()} standards, "
      f"{long_df['Known_Concentration'].isna().sum()} samples)")
print(f"Saved   : {LONG_CSV}")
long_df.head()


In [ ]:
# ============================================================================
# (2) Standard curve -> concentration for every well, saved to CSV
# ============================================================================
import matplotlib.pyplot as plt
from scipy.stats import linregress

# Standards above this raw value are dropped from the fit (top-end saturation).
# Set to None to fit the full series.
FIT_MAX_RAW = None

all_wells = long_df.copy()
std = all_wells[all_wells["Known_Concentration"].notna()].copy()

fit_set = std.dropna(subset=["Raw_Data", "Known_Concentration"])
if FIT_MAX_RAW is not None:
    fit_set = fit_set[fit_set["Raw_Data"] <= FIT_MAX_RAW]

slope, intercept, r_value, p_value, std_err = linregress(
    fit_set["Raw_Data"], fit_set["Known_Concentration"])

print(f"Standard curve (n={len(fit_set)}): "
      f"conc = {slope:.6g} * raw + {intercept:.6g}   R^2 = {r_value**2:.5f}")

# Apply to every well, standards included (back-calculated = QC on the curve).
all_wells["Concentration (ng/uL)"] = intercept + slope * all_wells["Raw_Data"]
n_neg = (all_wells["Concentration (ng/uL)"] < 0).sum()
all_wells["Concentration (ng/uL)"] = all_wells["Concentration (ng/uL)"].clip(lower=0)
if n_neg:
    print(f"{n_neg} well(s) read below the fitted blank and were clipped to 0.")

all_wells["Type"] = np.where(all_wells["Known_Concentration"].notna(),
                             "Standard", "Sample")

# Flag wells read outside the bracket of the standards: those concentrations are
# extrapolated off the end of the curve, not interpolated, so treat them as
# lower bounds and re-run diluted if the number matters.
raw_lo, raw_hi = fit_set["Raw_Data"].min(), fit_set["Raw_Data"].max()
all_wells["In_Std_Range"] = all_wells["Raw_Data"].between(raw_lo, raw_hi)
n_over = ((~all_wells["In_Std_Range"]) & (all_wells["Raw_Data"] > raw_hi)
          & (all_wells["Type"] == "Sample")).sum()
if n_over:
    top_conc = fit_set.loc[fit_set["Raw_Data"].idxmax(), "Known_Concentration"]
    print(f"WARNING: {n_over}/{(all_wells['Type'] == 'Sample').sum()} sample wells "
          f"read above the top standard ({top_conc:g} ng/uL, raw {raw_hi:g}); "
          f"their concentrations are extrapolated.")

out_cols = ["Well", "Row", "Col", "Type", "Content", "Sample_Name",
            "Raw_Data", "Known_Concentration", "Concentration (ng/uL)",
            "In_Std_Range"]
out_tidy = all_wells[out_cols].rename(
    columns={"Known_Concentration": "Known_Concentration (ng/uL)",
             "Raw_Data": "Raw Data (485-8/530-8)"})

stem = os.path.splitext(os.path.basename(PLATE_FILE))[0]
tidy_csv  = os.path.join(output_dir, f"{stem}_concentrations.csv")
plate_csv = os.path.join(output_dir, f"{stem}_concentrations_plate.csv")

out_tidy.to_csv(tidy_csv, index=False)

# Same numbers in 96-well plate shape, for quick eyeballing against the bench.
all_wells.pivot(index="Row", columns="Col",
                values="Concentration (ng/uL)").to_csv(plate_csv)

print(f"Saved: {tidy_csv}")
print(f"Saved: {plate_csv}")

# --- Standard curve plot -----------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))
smp = all_wells[all_wells["Type"] == "Sample"]
ax.scatter(smp["Raw_Data"], smp["Concentration (ng/uL)"],
           color="green", s=30, alpha=0.6, zorder=2, label="Samples (inferred)")
ax.scatter(std["Raw_Data"], std["Known_Concentration"],
           color="blue", s=50, zorder=3, label="Standards (known)")
xs = np.linspace(0, all_wells["Raw_Data"].max(), 100)
ax.plot(xs, intercept + slope * xs, color="black",
        label=f"Fit: R$^2$={r_value**2:.4f}")
ax.set_xlabel("Fluorescence (485-8/530-8)")
ax.set_ylabel("Concentration (ng/µL)")
ax.set_title(f"dsDNA BR standard curve — {plate_meta.get('ID1', stem)}")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{stem}_standard_curve.png"), dpi=300)
plt.show()

print(out_tidy.to_string(index=False))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from datetime import datetime

import os
_win_dir = r"C:\Users\mullenp\Desktop\Quant_Automated\Quantification"
if os.path.isdir(_win_dir):
    os.chdir(_win_dir)
print(f"Current working directory: {os.getcwd()}")

# create output folder
output_dir = os.path.join(os.getcwd(), "Output") if os.path.isdir(_win_dir) \
    else os.path.join(REPO, "concentrations")
os.makedirs(output_dir, exist_ok=True)

# Load data from qubit output CSV
data = pd.read_csv(LONG_CSV)   # <- reformatted plate data from cell (1)

# Ensure Raw_Data is numeric
data['Raw_Data'] = pd.to_numeric(data['Raw_Data'], errors='coerce')

# Extract standards — their Concentration column contains numeric known concentrations
standards = data[data['Content'].str.contains('Standard', na=False)].copy()
standards['Concentration'] = pd.to_numeric(standards['Concentration'], errors='coerce')
standards = standards.dropna(subset=['Raw_Data', 'Concentration'])

# Extract samples — their Concentration column contains sample IDs (e.g. 'gDNA_1'), not numbers.
# Populate Sample_Name from it if the dedicated column is empty.
samples = data[data['Content'].str.contains('Sample', na=False)].copy()
samples['Sample_Name'] = samples['Sample_Name'].fillna(samples['Concentration'])
samples['Sample_Name'] = samples['Sample_Name'].fillna('').astype(str).str.strip()
unnamed_mask = samples['Sample_Name'].str.lower().isin(['', 'nan', 'none'])
samples.loc[unnamed_mask, 'Sample_Name'] = 'Sample_' + samples.index[unnamed_mask].astype(str)

# Perform linear regression to get standard curve
slope, intercept, r_value, p_value, std_err = linregress(standards['Raw_Data'], standards['Concentration'])

# Infer concentrations from the standard curve.
samples['Concentration'] = intercept + slope * samples['Raw_Data']

# Use the average blank as baseline and set that baseline to 0 ng/uL.
blank_mask = samples['Sample_Name'].str.contains('Blank', case=False, na=False)
if blank_mask.any():
    blank_baseline = samples.loc[blank_mask, 'Concentration'].mean()
    samples['Concentration'] = (samples['Concentration'] - blank_baseline).clip(lower=0)
else:
    blank_baseline = 0.0
    samples['Concentration'] = samples['Concentration'].clip(lower=0)

print(f"Blank baseline subtracted: {blank_baseline:.4f} ng/uL")

# Plot standard curve
plt.figure(figsize=(8,6))
plt.scatter(standards['Raw_Data'], standards['Concentration'], color='blue', label='Standards', s=50, zorder=3)
plt.scatter(samples['Raw_Data'], samples['Concentration'], color='green', label='Samples', s=50, zorder=3)
plt.plot(standards['Raw_Data'], intercept + slope*standards['Raw_Data'], color='black', label=f'Fit: R²={r_value**2:.3f}')
plt.xlabel('Fluorescence (Raw_Data)')
plt.ylabel('Concentration')
plt.title('Qubit Standard Curve')
plt.yscale('linear')
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(output_dir, 'DNA_curve.png'), dpi=300)
plt.legend()
plt.show()

# Add bar plot of sample concentrations with color coding
plt.figure(figsize=(10,6))
colors = ['red' if conc < 0.5 else 'blue' for conc in samples['Concentration']]
x_labels = samples['Sample_Name'].where(samples['Sample_Name'].notna(), '').map(str).tolist()
plt.bar(x_labels, samples['Concentration'].to_numpy(), color=colors)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Sample')
plt.ylabel('Concentration')
plt.title('DNA Concentrations Across Samples')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'DNA_barplot.png'), dpi=300)
plt.show()

# Print sample calculated concentrations
print(samples[['Sample_Name', 'Raw_Data', 'Concentration']])

data['Concentration'] = np.nan
data.loc[samples.index, 'Concentration'] = samples['Concentration'].values

# Rename the column before exporting
data = data.rename(columns={'Raw_Data': 'Raw Data (625-15/665-15)'})
data = data.rename(columns={'Concentration': 'Inferred_Concentration (ng/uL)'})

# Export data: raw file
out_file = os.path.join(output_dir, 'DNA_raw_data.csv')
try:
    data.to_csv(out_file, index=False)
    print(f"Saved: {out_file}")
except PermissionError:
    # Fallback if the default output file is open/locked by another program.
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    fallback_file = os.path.join(output_dir, f'DNA_raw_data_{timestamp}.csv')
    data.to_csv(fallback_file, index=False)
    print(f"Default file is locked. Saved instead: {fallback_file}")


In [ ]:
import re

# Strip whitespace from sample names to avoid matching failures
samples['Sample_Name'] = samples['Sample_Name'].str.strip()

# --- Soil samples (no spike): name contains "Soil" ---
soil_only = samples[samples['Sample_Name'].str.contains('Soil', case=False, na=False)].copy()
soil_avg = soil_only['Concentration'].mean()

# --- Parse spike samples ---
# Naming scheme:
#   {amount}ngRD          → RNA+DNA numeric spike, lysis (L) step
#   {amount}ngRD_ISO      → RNA+DNA numeric spike, ISO step
#   {amount}ngR[_ISO]     → RNA-only control
#   {amount}ngD[_ISO]     → DNA-only control
#   {r}ngR_{d}ngD[_ISO]   → combined RNA+DNA spike

def parse_spike_info(name):
    """Returns (step, spike_ng, amount_code), or (None, None, None) if unrecognised."""
    name = name.strip()
    step = 'ISO' if name.endswith('_ISO') else 'L'
    base = name[:-4] if name.endswith('_ISO') else name

    m = re.match(r'^(\d+\.?\d*)ngRD$', base)          # e.g. 6.25ngRD
    if m:
        return step, float(m.group(1)), 'RD'

    m = re.match(r'^(\d+\.?\d*)ngR$', base)            # e.g. 200ngR
    if m:
        return step, float(m.group(1)), 'R'

    m = re.match(r'^(\d+\.?\d*)ngD$', base)            # e.g. 100ngD
    if m:
        return step, float(m.group(1)), 'D'

    m = re.match(r'^(\d+\.?\d*)ngR_(\d+\.?\d*)ngD$', base)   # e.g. 200ngR_100ngD
    if m:
        return step, float(m.group(2)), 'RD_combo'  # x-axis = DNA amount only

    return None, None, None

# Match any recognised spike pattern (exclude plain Soil names)
spike_mask = samples['Sample_Name'].apply(
    lambda n: parse_spike_info(n)[0] is not None and 'Soil' not in n
)
spike_samples = samples[spike_mask].copy()

if spike_samples.empty:
    print("WARNING: No spike samples found. Skipping spike curve plots.")
    print(f"Sample names in dataset: {samples['Sample_Name'].tolist()}")
    numeric_spikes = pd.DataFrame()
else:
    spike_samples[['Step', 'Spike_ng', 'Amount_Code']] = spike_samples['Sample_Name'].apply(
        lambda n: pd.Series(parse_spike_info(n))
    )

    # Numeric spikes: ngRD and RD_combo (100ng DNA); letter-only codes = controls
    numeric_spikes = spike_samples[~spike_samples['Amount_Code'].isin(['R', 'D'])].copy()
    controls       = spike_samples[spike_samples['Amount_Code'].isin(['R', 'D'])].copy()

    step_colors = {'L': 'steelblue', 'ISO': 'darkorange'}
    # Small multiplicative dodge so L and ISO points don't overlap on the log x-axis
    step_dodge  = {'L': 0.92, 'ISO': 1.08}
    steps = numeric_spikes['Step'].unique()

    fig, ax = plt.subplots(figsize=(9, 6))

    for step in sorted(steps):
        grp = numeric_spikes[numeric_spikes['Step'] == step]
        col   = step_colors.get(step, 'gray')
        dodge = step_dodge.get(step, 1.0)
        ax.scatter(grp['Spike_ng'] * dodge, grp['Concentration'],
                   color=col, alpha=0.7, zorder=3, label=f'{step} step')

    # Soil background reference line
    if not np.isnan(soil_avg):
        ax.axhline(soil_avg, color='saddlebrown', linestyle='--', linewidth=1.5,
                   label=f'Avg Soil (no spike): {soil_avg:.2f} ng/µL')

    # Controls as distinct markers
    ctrl_styles = {'R': ('RNA ctrl', '^', 'green'),
                   'D': ('DNA ctrl', 's', 'purple')}
    for code, (lbl, mk, col) in ctrl_styles.items():
        c = controls[controls['Amount_Code'] == code]
        if not c.empty:
            for step in sorted(c['Step'].unique()):
                sub = c[c['Step'] == step]
                ax.scatter(sub['Spike_ng'], sub['Concentration'],
                           color=col, marker=mk, s=80, zorder=5,
                           label=f'{lbl} ({step} step)')

    ax.set_xscale('log')
    ax.set_xlabel('DNA Spiked In (ng)')
    ax.set_ylabel('Total DNA Extracted (ng/µL)')
    ax.set_title('Total DNA Extracted vs. DNA Spiked In')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'DNA_spike_curve.png'), dpi=300)
    plt.show()


Sample names in dataset: ['S_1', 'S_2', 'S_3', 'S_4', 'S_5', 'S_6', 'S_7', 'S_8', 'S_9']


In [ ]:
# --- Extraction Efficiency Plot ---
# Recovery (ng/µL per ng spiked) = (measured - soil background) / ng spiked in
# A flat line = constant recovery; declining = losses at higher spike amounts

if spike_samples.empty or numeric_spikes.empty:
    print("WARNING: No numeric spike samples found. Skipping efficiency plot.")
else:
    eff = numeric_spikes.copy()
    eff['Background_Subtracted'] = eff['Concentration'] - soil_avg
    eff['Recovery'] = eff['Background_Subtracted'] / eff['Spike_ng']

    step_colors = {'L': 'steelblue', 'ISO': 'darkorange'}
    steps = eff['Step'].unique()

    fig, ax = plt.subplots(figsize=(9, 6))

    for step in sorted(steps):
        grp = eff[eff['Step'] == step]
        col = step_colors.get(step, 'gray')

        ax.scatter(grp['Spike_ng'], grp['Recovery'],
                   color=col, alpha=0.6, zorder=3, label=f'{step} step')

    ax.axhline(0, color='gray', linestyle=':', linewidth=1)
    ax.set_xscale('log')
    ax.set_xlabel('DNA Spiked In (ng)')
    ax.set_ylabel('Recovery (ng/µL per ng spiked)')
    ax.set_title('DNA Extraction Efficiency vs. Amount Spiked In\n(background-subtracted)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'DNA_efficiency.png'), dpi=300)
    plt.show()

    print("Recovery (ng/µL per ng spiked) per replicate:")
    summary = eff[['Step', 'Spike_ng', 'Sample_Name', 'Recovery']].sort_values(['Step', 'Spike_ng'])
    summary.columns = ['Step', 'Spike_ng (ng)', 'Sample', 'Recovery (ng/µL per ng)']
    print(summary.to_string(index=False))


In [ ]:
print("Unspiked soil replicates:")
for _, row in soil_only.iterrows():
    print(f"  {row['Sample_Name']}: {row['Concentration']:.4f} ng/µL")
print(f"Mean: {soil_avg:.4f} ng/µL")


Unspiked soil replicates:
Mean: nan ng/µL


In [ ]:
# ============================================================================
# (7) 10/20 run: blanks + batch 1 + batch 3  ->  concentrations
# ============================================================================
# Needs read_bmg_plate() from cell (1); otherwise independent of cells (2)-(6),
# which stay pointed at the 10/16 soil-practice plate.
#
# Layout of 20261020_Ik_DNA_BR_blank_batch1_batch3.CSV:
#   col 1       A,B = blanks;  C,D,E = batch 1;  F,G,H = batch 3
#   cols 11,12  8-point standard curve, 100 ng/uL halving down to a 0 blank
#               (the two columns are replicates, already averaged by the reader)
# Well -> name / known-concentration assignments live in the *_metadata.csv pair.
from scipy.stats import linregress

PLATE_1020 = os.path.join(REPO, "absorbances",
                          "20261020_Ik_DNA_BR_blank_batch1_batch3.CSV")
STD_META_1020 = os.path.join(REPO, "absorbances", "20261020_DNA_standards_metadata.csv")
SMP_META_1020 = os.path.join(REPO, "absorbances", "20261020_DNA_samples_metadata.csv")

plate_1020, meta_1020 = read_bmg_plate(PLATE_1020)
plate_1020 = plate_1020.dropna(subset=["Raw_Data"])        # drop the unused columns

wells = (plate_1020
         .merge(pd.read_csv(STD_META_1020).rename(
                    columns={"Concentration": "Known_Concentration"}),
                on="Well", how="left")
         .merge(pd.read_csv(SMP_META_1020), on="Well", how="left"))
wells["Type"] = np.where(wells["Known_Concentration"].notna(), "Standard", "Sample")

# --- standard curve --------------------------------------------------------
std_1020 = wells[wells["Type"] == "Standard"]
fit = linregress(std_1020["Raw_Data"], std_1020["Known_Concentration"])
print(f"Run          : {meta_1020.get('Test name')} / {meta_1020.get('ID1')} "
      f"({meta_1020.get('Date')} {meta_1020.get('Time')})")
print(f"Standard curve (n={std_1020['Well'].nunique()} wells): "
      f"conc = {fit.slope:.6g} * raw + {fit.intercept:.6g}   R^2 = {fit.rvalue**2:.5f}")

wells["Concentration (ng/uL)"] = fit.intercept + fit.slope * wells["Raw_Data"]

# --- blanks define zero ----------------------------------------------------
smp_1020 = wells[wells["Type"] == "Sample"].copy()
is_blank = smp_1020["Sample_Name"].str.startswith("Blank")
blank_baseline = smp_1020.loc[is_blank, "Concentration (ng/uL)"].mean()
smp_1020["Concentration (ng/uL)"] -= blank_baseline
print(f"Blank baseline subtracted: {blank_baseline:.4f} ng/uL "
      f"(wells {', '.join(smp_1020.loc[is_blank, 'Well'])})")

# Readings above the top standard are extrapolated off the end of the curve,
# so they are lower bounds -- re-run diluted if the exact number matters.
raw_hi = std_1020["Raw_Data"].max()
over = smp_1020[(~is_blank) & (smp_1020["Raw_Data"] > raw_hi)]
if not over.empty:
    top = std_1020.loc[std_1020["Raw_Data"].idxmax(), "Known_Concentration"]
    print(f"WARNING: {len(over)} sample(s) read above the top standard "
          f"({top:g} ng/uL, raw {raw_hi:g}) and are extrapolated: "
          f"{', '.join(over['Sample_Name'])}")

smp_1020["Batch"] = np.where(is_blank, "Blank",
                      np.where(smp_1020["Sample_Name"].str.startswith("Batch1"),
                               "Batch 1", "Batch 3"))

stem_1020 = os.path.splitext(os.path.basename(PLATE_1020))[0]
csv_1020 = os.path.join(output_dir, f"{stem_1020}_concentrations.csv")
smp_1020[["Well", "Sample_Name", "Batch", "Raw_Data", "Concentration (ng/uL)"]] \
    .to_csv(csv_1020, index=False)
print(f"Saved: {csv_1020}\n")
print(smp_1020[["Well", "Sample_Name", "Batch", "Raw_Data",
                "Concentration (ng/uL)"]].to_string(index=False))

In [ ]:
# ============================================================================
# (8) Batch 1 vs batch 3 DNA  --  mean +/- SD with the three replicates on top
# ============================================================================
import matplotlib.pyplot as plt

ERROR = "sd"          # "sd" = spread of the 3 replicates, "sem" = sd/sqrt(n)

BATCH_ORDER  = ["Batch 1", "Batch 3"]
BATCH_COLORS = {"Batch 1": "#2a78d6", "Batch 3": "#eb6834"}
INK, INK_MUTED, SURFACE = "#0b0b0b", "#52514e", "#ffffff"

batches = (smp_1020[smp_1020["Batch"] != "Blank"]
           .groupby("Batch")["Concentration (ng/uL)"]
           .agg(mean="mean", sd=lambda s: s.std(ddof=1), n="size")
           .reindex(BATCH_ORDER))
batches["sem"] = batches["sd"] / np.sqrt(batches["n"])
err = batches[ERROR]

fig, ax = plt.subplots(figsize=(5.5, 4.2))
x = np.arange(len(batches))

ax.bar(x, batches["mean"], width=0.55,
       color=[BATCH_COLORS[b] for b in batches.index],
       edgecolor=SURFACE, linewidth=2, zorder=2)
ax.errorbar(x, batches["mean"], yerr=err, fmt="none",
            ecolor=INK_MUTED, elinewidth=1.5, capsize=6, capthick=1.5, zorder=3)

# Individual replicates, dodged right so n=3 is visible and the dots never
# sit on top of the error bar.
tops = []
for i, b in enumerate(batches.index):
    pts = smp_1020.loc[smp_1020["Batch"] == b, "Concentration (ng/uL)"]
    ax.scatter(np.full(len(pts), i + 0.17), pts,
               s=40, color=INK, edgecolor=SURFACE, linewidth=1.5, zorder=4)
    tops.append(max(pts.max(), batches["mean"].iloc[i] + err.iloc[i]))

for i, (m, top) in enumerate(zip(batches["mean"], tops)):
    ax.text(i, top + 4, f"{m:.1f}", ha="center", va="bottom",
            fontsize=11, color=INK)

ax.set_xticks(x)
ax.set_xticklabels([f"{b}\n(n={int(n)})" for b, n in zip(batches.index, batches["n"])],
                   color=INK)
ax.set_ylabel("DNA concentration (ng/µL)", color=INK_MUTED, fontsize=10)
ax.set_title(f"Extracted DNA by batch  —  mean ± {ERROR.upper()}, dots = replicates",
             color=INK, fontsize=11, loc="left")
ax.set_ylim(0, max(tops) * 1.20)
ax.yaxis.grid(True, color="#e6e5e1", linewidth=1)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.spines["left"].set_color("#e6e5e1")
ax.spines["bottom"].set_color("#e6e5e1")
ax.tick_params(colors=INK_MUTED, length=0)

plt.tight_layout()
png_1020 = os.path.join(output_dir, f"{stem_1020}_batch_barplot.png")
plt.savefig(png_1020, dpi=300)
plt.show()
print(f"Saved: {png_1020}\n")
print(batches.rename(columns={"mean": "Mean (ng/uL)", "sd": "SD", "sem": "SEM"})
             .to_string(float_format=lambda v: f"{v:.3f}"))

In [ ]:
# ============================================================================
# (9) 10/21 run: batch 1 + batch 4  ->  DNA concentration in every well
# ============================================================================
# Needs read_bmg_plate() from cell (1).
#
# Standard curve : 20261021_16-30-58_Ik_DNA_BR_standard.CSV (A-H x cols 1-3,
#                  100 -> 0 in 2-fold steps, 3 replicates each)
# Samples        : 2nd read of each batch plate (16:34:22 batch 1, 16:38:45 batch 4)
# Well -> known concentration / sample name come from the 20261021 *_metadata.csv
# pair; the samples metadata is shared by both batch plates.
# Concentrations are in the units of the standards metadata, with no dilution
# or volume correction.
from scipy.stats import linregress

ABS_1021 = os.path.join(REPO, "absorbances")
STD_PLATE_1021 = os.path.join(ABS_1021, "20261021_16-30-58_Ik_DNA_BR_standard.CSV")
SMP_PLATES_1021 = {
    "Batch 1": os.path.join(ABS_1021, "20261021_16-34-22_Ik_DNA_BR_batch1.CSV"),
    "Batch 4": os.path.join(ABS_1021, "20261021_16-38-45_Ik_DNA_BR_batch4.CSV"),
}
STD_META_1021 = os.path.join(ABS_1021, "20261021_DNA_standards_metadata.csv")
SMP_META_1021 = os.path.join(ABS_1021, "20261021_DNA_samples_metadata.csv")

# --- standard curve --------------------------------------------------------
std_plate_1021, std_hdr_1021 = read_bmg_plate(STD_PLATE_1021)
std_1021 = (std_plate_1021.dropna(subset=["Raw_Data"])
            .merge(pd.read_csv(STD_META_1021).rename(
                       columns={"Concentration": "Known_Concentration"}),
                   on="Well", how="inner"))
fit_1021 = linregress(std_1021["Raw_Data"], std_1021["Known_Concentration"])
print(f"Standards    : {std_hdr_1021.get('ID1')} ({std_hdr_1021.get('Date')} "
      f"{std_hdr_1021.get('Time')})")
print(f"Standard curve (n={len(std_1021)} wells): conc = {fit_1021.slope:.6g} * raw "
      f"+ {fit_1021.intercept:.6g}   R^2 = {fit_1021.rvalue**2:.5f}")

# --- samples ---------------------------------------------------------------
smp_meta_1021 = pd.read_csv(SMP_META_1021)
raw_hi_1021 = std_1021["Raw_Data"].max()
frames = []
for batch, path in SMP_PLATES_1021.items():
    plate, hdr = read_bmg_plate(path)
    df = (plate.dropna(subset=["Raw_Data"])
          .merge(smp_meta_1021, on="Well", how="left"))
    df["Batch"] = batch
    df["Concentration"] = fit_1021.intercept + fit_1021.slope * df["Raw_Data"]

    # blank wells define zero for this plate
    is_blank = df["Sample_type"] == "Blank"
    baseline = df.loc[is_blank, "Concentration"].mean()
    df["Concentration"] -= baseline
    print(f"{batch}: {hdr.get('Time')}  blank baseline {baseline:.3f} subtracted "
          f"(wells {', '.join(df.loc[is_blank, 'Well'])})")

    over = df[(~is_blank) & (df["Raw_Data"] > raw_hi_1021)]
    if not over.empty:
        print(f"WARNING: {batch}: {len(over)} well(s) above the top standard "
              f"(raw {raw_hi_1021:g}), extrapolated: {', '.join(over['Well'])}")
    frames.append(df)

dna_1021 = pd.concat(frames, ignore_index=True)
cols_1021 = ["Well", "Sample_type", "Nitrite_input", "Nitrate_input",
             "Chloramphenicol", "Raw_Data", "Concentration"]
for batch, df in dna_1021.groupby("Batch"):
    out_1021 = os.path.join(
        output_dir, f"total_DNA_batch{batch.split()[-1]}_20261021.csv")
    df[cols_1021].to_csv(out_1021, index=False)
    print(f"Saved: {out_1021}")
print()

# plate-shaped view, one block per batch
for batch, df in dna_1021.groupby("Batch"):
    print(batch)
    print(df.pivot(index="Row", columns="Col", values="Concentration")
            .to_string(float_format=lambda v: f"{v:7.2f}"), "\n")